# Notebook 08: End-to-End Pipeline Orchestration

## Purpose

This notebook executes the complete PaySim pipeline using the reusable modules
created in Notebook 07.

The pipeline performs:

1. configuration validation;
2. Spark-session initialization;
3. Bronze ingestion;
4. Bronze validation;
5. Silver transformation;
6. Silver validation;
7. Gold table construction;
8. cross-layer reconciliation;
9. output persistence;
10. audit-record generation;
11. pipeline completion reporting.

## Design principle

This notebook controls execution. Transformation logic remains inside
`src/paysim_pipeline/`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

import sys
import uuid

import pandas as pd

In [2]:
current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print("Current directory:", current_path)
print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)

Current directory: C:\Projects\paysim-financial-data-pipeline\notebooks
Project root: C:\Projects\paysim-financial-data-pipeline
Source path: C:\Projects\paysim-financial-data-pipeline\src


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
%reload_ext autoreload

In [5]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

In [6]:
from paysim_pipeline.config import PipelineConfig
from paysim_pipeline.spark_session import (
    create_spark_session,
)

from paysim_pipeline.bronze import (
    build_bronze_transactions,
)

from paysim_pipeline.silver import (
    build_silver_transactions,
)

from paysim_pipeline.gold import (
    build_gold_tables,
)

from paysim_pipeline.schemas import (
    BRONZE_REQUIRED_COLUMNS,
    SILVER_REQUIRED_COLUMNS,
)

from paysim_pipeline.validation import (
    build_null_profile,
    validate_required_columns,
)

from paysim_pipeline.audit import (
    combine_audit_records,
    create_audit_record,
)

from paysim_pipeline.io_utils import (
    clear_csv_outputs,
    export_small_dataframe_to_csv,
    write_dataframe_to_parquet,
)

In [7]:
RAW_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

print("Raw CSV path:", RAW_CSV_PATH)
print("Raw CSV exists:", RAW_CSV_PATH.exists())

Raw CSV path: C:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
Raw CSV exists: True


In [8]:
PIPELINE_PROFILE = "local"

CLEAR_EXISTING_SUMMARY_CSVS = True
WRITE_LARGE_GOLD_TABLES = False
EXPORT_SMALL_GOLD_TABLES = True
RUN_NULL_PROFILES = True

In [9]:
config = PipelineConfig(
    project_root=PROJECT_ROOT,
    raw_csv_path=RAW_CSV_PATH,
    application_name="PaySimEndToEndPipeline",
    spark_master="local[4]",
    driver_memory="8g",
    shuffle_partitions=64,
    high_value_threshold=200_000.0,
    approximate_distinct_rsd=0.05,
)

config.create_directories()
config.validate()

print("Pipeline configuration validated.")

Pipeline configuration validated.


In [10]:
run_started_at = datetime.now(timezone.utc)

run_timestamp = run_started_at.strftime(
    "%Y%m%d_%H%M%S"
)

pipeline_run_id = (
    f"{run_timestamp}_{uuid.uuid4().hex[:8]}"
)

print("Pipeline run ID:", pipeline_run_id)
print("Pipeline start time:", run_started_at)

Pipeline run ID: 20260726_003734_d826931e
Pipeline start time: 2026-07-26 00:37:34.094658+00:00


In [11]:
SMALL_GOLD_TABLES = [
    "daily_transaction_summary",
    "daily_type_summary",
    "hourly_fraud_summary",
    "transaction_type_summary",
    "high_value_summary",
]

TRANSACTION_LEVEL_GOLD_TABLES = [
    "fraud_monitoring",
]

LARGE_GOLD_TABLES = [
    "origin_account_summary",
    "destination_account_summary",
    "fraud_feature",
]

print("Small Gold tables:", SMALL_GOLD_TABLES)
print("Transaction-level tables:", TRANSACTION_LEVEL_GOLD_TABLES)
print("Large Gold tables:", LARGE_GOLD_TABLES)

Small Gold tables: ['daily_transaction_summary', 'daily_type_summary', 'hourly_fraud_summary', 'transaction_type_summary', 'high_value_summary']
Transaction-level tables: ['fraud_monitoring']
Large Gold tables: ['origin_account_summary', 'destination_account_summary', 'fraud_feature']


In [12]:
if CLEAR_EXISTING_SUMMARY_CSVS:
    deleted_files = clear_csv_outputs(
        config.gold_summary_output_path
    )

    print(
        "Deleted previous summary CSV files:",
        len(deleted_files),
    )

    for deleted_file in deleted_files:
        print("Deleted:", deleted_file.name)
else:
    print("Previous summary CSV files retained.")

Deleted previous summary CSV files: 1
Deleted: modular_transaction_type_summary_20260726_002738_a2366ff7.csv


In [14]:
spark = create_spark_session(
    config=config,
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Driver memory:",
    spark.conf.get("spark.driver.memory"),
)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions"),
)
print(
    "Spark local directory:",
    spark.conf.get("spark.local.dir"),
)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark master: local[4]
Driver memory: 8g
Shuffle partitions: 64
Spark local directory: C:\Projects\paysim-financial-data-pipeline\spark-temp


In [15]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [16]:
stage_metrics = {}
audit_dataframes = []
exported_files = []
pipeline_status = "RUNNING"

pipeline_start_timer = perf_counter()

print("Execution tracking initialized.")

Execution tracking initialized.


In [17]:
bronze_stage_start = perf_counter()

bronze_df = build_bronze_transactions(
    spark=spark,
    raw_csv_path=config.raw_csv_path,
    pipeline_run_id=pipeline_run_id,
)

bronze_execution_seconds = (
    perf_counter() - bronze_stage_start
)

print(
    "Bronze DataFrame defined in:",
    round(bronze_execution_seconds, 2),
    "seconds",
)

Bronze DataFrame defined in: 0.21 seconds


In [18]:
validate_required_columns(
    dataframe=bronze_df,
    required_columns=BRONZE_REQUIRED_COLUMNS,
    dataframe_name="bronze_df",
)

print("Bronze required-column validation: PASS")

Bronze required-column validation: PASS


In [19]:
bronze_action_start = perf_counter()

bronze_row_count = bronze_df.count()

bronze_action_seconds = (
    perf_counter() - bronze_action_start
)

stage_metrics["bronze"] = {
    "row_count": bronze_row_count,
    "column_count": len(bronze_df.columns),
    "execution_time_seconds": (
        bronze_execution_seconds
        + bronze_action_seconds
    ),
}

print("Bronze row count:", f"{bronze_row_count:,}")
print(
    "Bronze execution time:",
    round(
        stage_metrics["bronze"][
            "execution_time_seconds"
        ],
        2,
    ),
    "seconds",
)

Bronze row count: 6,362,620
Bronze execution time: 1.4 seconds


In [20]:
bronze_df.select(
    "step",
    "transaction_type",
    "amount",
    "origin_account",
    "destination_account",
    "is_fraud",
    "is_flagged_fraud",
    "pipeline_run_id",
).show(
    n=5,
    truncate=False,
)

+----+----------------+--------+--------------+-------------------+--------+----------------+------------------------+
|step|transaction_type|amount  |origin_account|destination_account|is_fraud|is_flagged_fraud|pipeline_run_id         |
+----+----------------+--------+--------------+-------------------+--------+----------------+------------------------+
|1   |PAYMENT         |9839.64 |C1231006815   |M1979787155        |0       |0               |20260726_003734_d826931e|
|1   |PAYMENT         |1864.28 |C1666544295   |M2044282225        |0       |0               |20260726_003734_d826931e|
|1   |TRANSFER        |181.0   |C1305486145   |C553264065         |1       |0               |20260726_003734_d826931e|
|1   |CASH_OUT        |181.0   |C840083671    |C38997010          |1       |0               |20260726_003734_d826931e|
|1   |PAYMENT         |11668.14|C2048537720   |M1230701703        |0       |0               |20260726_003734_d826931e|
+----+----------------+--------+--------------+-

In [21]:
bronze_audit_df = create_audit_record(
    spark=spark,
    dataframe=bronze_df,
    pipeline_run_id=pipeline_run_id,
    pipeline_stage="BRONZE",
    table_name="bronze_transactions",
    execution_time_seconds=(
        stage_metrics["bronze"][
            "execution_time_seconds"
        ]
    ),
    reconciliation_status="PASS",
)

audit_dataframes.append(
    bronze_audit_df
)

print("Bronze audit record created.")

Bronze audit record created.


In [22]:
silver_stage_start = perf_counter()

silver_df = build_silver_transactions(
    bronze_dataframe=bronze_df,
    high_value_threshold=(
        config.high_value_threshold
    ),
)

silver_definition_seconds = (
    perf_counter() - silver_stage_start
)

print(
    "Silver DataFrame defined in:",
    round(silver_definition_seconds, 2),
    "seconds",
)

Silver DataFrame defined in: 0.39 seconds


In [23]:
validate_required_columns(
    dataframe=silver_df,
    required_columns=SILVER_REQUIRED_COLUMNS,
    dataframe_name="silver_df",
)

print("Silver required-column validation: PASS")

Silver required-column validation: PASS


In [24]:
silver_df = silver_df.persist(
    StorageLevel.DISK_ONLY
)

print("Silver storage level:", silver_df.storageLevel)

Silver storage level: Disk Serialized 1x Replicated


In [25]:
silver_action_start = perf_counter()

silver_row_count = silver_df.count()

silver_action_seconds = (
    perf_counter() - silver_action_start
)

silver_execution_seconds = (
    silver_definition_seconds
    + silver_action_seconds
)

stage_metrics["silver"] = {
    "row_count": silver_row_count,
    "column_count": len(silver_df.columns),
    "execution_time_seconds": (
        silver_execution_seconds
    ),
}

print("Silver row count:", f"{silver_row_count:,}")
print(
    "Silver execution time:",
    round(silver_execution_seconds, 2),
    "seconds",
)

Silver row count: 6,362,620
Silver execution time: 12.43 seconds


In [26]:
bronze_to_silver_difference = (
    bronze_row_count - silver_row_count
)

bronze_to_silver_status = (
    "PASS"
    if 0 <= silver_row_count <= bronze_row_count
    else "FAIL"
)

print("Bronze rows:", f"{bronze_row_count:,}")
print("Silver rows:", f"{silver_row_count:,}")
print(
    "Rows rejected from Silver:",
    f"{bronze_to_silver_difference:,}",
)
print(
    "Bronze-to-Silver reconciliation:",
    bronze_to_silver_status,
)

Bronze rows: 6,362,620
Silver rows: 6,362,620
Rows rejected from Silver: 0
Bronze-to-Silver reconciliation: PASS


In [27]:
silver_df.select(
    "step",
    "transaction_day",
    "transaction_hour",
    "transaction_type",
    "amount",
    "origin_account",
    "destination_account",
    "is_fraud",
    "is_high_value_transaction",
    "origin_balance_error",
    "destination_balance_error",
).show(
    n=10,
    truncate=False,
)

+----+---------------+----------------+----------------+--------+--------------+-------------------+--------+-------------------------+--------------------+-------------------------+
|step|transaction_day|transaction_hour|transaction_type|amount  |origin_account|destination_account|is_fraud|is_high_value_transaction|origin_balance_error|destination_balance_error|
+----+---------------+----------------+----------------+--------+--------------+-------------------+--------+-------------------------+--------------------+-------------------------+
|1   |1              |0               |PAYMENT         |9839.64 |C1231006815   |M1979787155        |0       |0                        |0.0                 |9839.64                  |
|1   |1              |0               |PAYMENT         |1864.28 |C1666544295   |M2044282225        |0       |0                        |0.0                 |1864.28                  |
|1   |1              |0               |TRANSFER        |181.0   |C1305486145   |C5532

In [28]:
silver_audit_df = create_audit_record(
    spark=spark,
    dataframe=silver_df,
    pipeline_run_id=pipeline_run_id,
    pipeline_stage="SILVER",
    table_name="silver_transactions",
    execution_time_seconds=(
        silver_execution_seconds
    ),
    reconciliation_status=(
        bronze_to_silver_status
    ),
)

audit_dataframes.append(
    silver_audit_df
)

print("Silver audit record created.")

Silver audit record created.


In [29]:
if RUN_NULL_PROFILES:
    bronze_null_profile_df = (
        build_null_profile(
            bronze_df.select(
                *BRONZE_REQUIRED_COLUMNS
            )
        )
    )

    bronze_null_profile_df.show(
        truncate=False
    )
else:
    print("Bronze null profile skipped.")

+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+
|step|transaction_type|amount|origin_account|origin_old_balance|origin_new_balance|destination_account|destination_old_balance|destination_new_balance|is_fraud|is_flagged_fraud|
+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+
|0   |0               |0     |0             |0                 |0                 |0                  |0                      |0                      |0       |0               |
+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+



In [30]:
if RUN_NULL_PROFILES:
    silver_null_profile_df = (
        build_null_profile(
            silver_df.select(
                *SILVER_REQUIRED_COLUMNS
            )
        )
    )

    silver_null_profile_df.show(
        truncate=False
    )
else:
    print("Silver null profile skipped.")

+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+---------------+----------------+-------------------------+--------------------+-------------------------+
|step|transaction_type|amount|origin_account|origin_old_balance|origin_new_balance|destination_account|destination_old_balance|destination_new_balance|is_fraud|is_flagged_fraud|transaction_day|transaction_hour|is_high_value_transaction|origin_balance_error|destination_balance_error|
+----+----------------+------+--------------+------------------+------------------+-------------------+-----------------------+-----------------------+--------+----------------+---------------+----------------+-------------------------+--------------------+-------------------------+
|0   |0               |0     |0             |0                 |0                 |0                  |0                      |0                    

In [31]:
binary_validation_df = (
    silver_df
    .agg(
        F.sum(
            F.when(
                ~F.col("is_fraud").isin(0, 1),
                1,
            ).otherwise(0)
        ).alias("invalid_is_fraud_count"),

        F.sum(
            F.when(
                ~F.col(
                    "is_flagged_fraud"
                ).isin(0, 1),
                1,
            ).otherwise(0)
        ).alias(
            "invalid_is_flagged_fraud_count"
        ),

        F.sum(
            F.when(
                ~F.col(
                    "is_high_value_transaction"
                ).isin(0, 1),
                1,
            ).otherwise(0)
        ).alias(
            "invalid_high_value_count"
        ),
    )
)

binary_validation_df.show(
    truncate=False
)

+----------------------+------------------------------+------------------------+
|invalid_is_fraud_count|invalid_is_flagged_fraud_count|invalid_high_value_count|
+----------------------+------------------------------+------------------------+
|0                     |0                             |0                       |
+----------------------+------------------------------+------------------------+



In [32]:
gold_definition_start = perf_counter()

gold_tables = build_gold_tables(
    silver_dataframe=silver_df,
    approximate_distinct_rsd=(
        config.approximate_distinct_rsd
    ),
)

gold_definition_seconds = (
    perf_counter() - gold_definition_start
)

print(
    "Gold transformation plans defined in:",
    round(gold_definition_seconds, 2),
    "seconds",
)

print("Gold tables:")

for table_name in sorted(gold_tables):
    print("-", table_name)

Gold transformation plans defined in: 0.42 seconds
Gold tables:
- daily_transaction_summary
- daily_type_summary
- destination_account_summary
- fraud_feature
- fraud_monitoring
- high_value_summary
- hourly_fraud_summary
- origin_account_summary
- transaction_type_summary


In [33]:
EXPECTED_GOLD_TABLES = {
    "daily_transaction_summary",
    "daily_type_summary",
    "hourly_fraud_summary",
    "transaction_type_summary",
    "origin_account_summary",
    "destination_account_summary",
    "high_value_summary",
    "fraud_monitoring",
    "fraud_feature",
}

actual_gold_tables = set(
    gold_tables.keys()
)

missing_gold_tables = (
    EXPECTED_GOLD_TABLES
    - actual_gold_tables
)

unexpected_gold_tables = (
    actual_gold_tables
    - EXPECTED_GOLD_TABLES
)

print("Missing Gold tables:", missing_gold_tables)
print(
    "Unexpected Gold tables:",
    unexpected_gold_tables,
)

assert not missing_gold_tables
assert not unexpected_gold_tables

print("Gold table registry validation: PASS")

Missing Gold tables: set()
Unexpected Gold tables: set()
Gold table registry validation: PASS


In [34]:
gold_row_counts = {}
gold_execution_times = {}

for table_name in SMALL_GOLD_TABLES:
    print(f"Executing: {table_name}")

    table_start = perf_counter()

    gold_tables[table_name] = (
        gold_tables[table_name]
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

    table_row_count = (
        gold_tables[table_name].count()
    )

    table_seconds = (
        perf_counter() - table_start
    )

    gold_row_counts[table_name] = (
        table_row_count
    )

    gold_execution_times[table_name] = (
        table_seconds
    )

    print(
        f"{table_name}: "
        f"{table_row_count:,} rows, "
        f"{table_seconds:.2f} seconds"
    )

Executing: daily_transaction_summary
daily_transaction_summary: 31 rows, 4.79 seconds
Executing: daily_type_summary
daily_type_summary: 152 rows, 3.09 seconds
Executing: hourly_fraud_summary
hourly_fraud_summary: 743 rows, 2.85 seconds
Executing: transaction_type_summary
transaction_type_summary: 5 rows, 0.97 seconds
Executing: high_value_summary
high_value_summary: 103 rows, 2.62 seconds


In [35]:
gold_tables[
    "daily_transaction_summary"
].show(
    n=40,
    truncate=False,
)

+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------------------------+-------------------------------------+--------------+----------------+
|transaction_day|transaction_count|total_transaction_amount|average_transaction_amount|minimum_transaction_amount|maximum_transaction_amount|fraud_count|fraud_amount  |flagged_fraud_count|high_value_transaction_count|estimated_unique_origin_accounts|estimated_unique_destination_accounts|fraud_rate_pct|fraud_amount_pct|
+---------------+-----------------+------------------------+--------------------------+--------------------------+--------------------------+-----------+--------------+-------------------+----------------------------+--------------------------------+-------------------------------------+--------------+----------------+
|1              |574255           |9.

In [36]:
gold_tables[
    "transaction_type_summary"
].show(
    truncate=False,
)

+----------------+-----------------+------------------------+--------------------------+-----------+-------------------+----------------------------+--------------+
|transaction_type|transaction_count|total_transaction_amount|average_transaction_amount|fraud_count|flagged_fraud_count|high_value_transaction_count|fraud_rate_pct|
+----------------+-----------------+------------------------+--------------------------+-----------+-------------------+----------------------------+--------------+
|CASH_IN         |1399284          |2.3636739191246E11      |168920.24                 |0          |0                  |475868                      |0.0           |
|CASH_OUT        |2237500          |3.9441299522449E11      |176273.96                 |4116       |0                  |788559                      |0.183955      |
|DEBIT           |41432            |2.2719922128E8          |5483.67                   |0          |0                  |27                          |0.0           |
|PAYMENT  

In [37]:
gold_tables[
    "high_value_summary"
].show(
    n=20,
    truncate=False,
)

+---------------+----------------+----------------------------+-----------------------+-------------------------+----------------------+
|transaction_day|transaction_type|high_value_transaction_count|high_value_total_amount|high_value_average_amount|high_value_fraud_count|
+---------------+----------------+----------------------------+-----------------------+-------------------------+----------------------+
|1              |CASH_IN         |42946                       |1.358067710166E10      |316226.82                |0                     |
|1              |CASH_OUT        |76731                       |2.522775263348E10      |328781.75                |65                    |
|1              |TRANSFER        |36073                       |3.012066657812E10      |834992.0                 |62                    |
|2              |CASH_IN         |33613                       |1.050648605166E10      |312572.1                 |0                     |
|2              |CASH_OUT        |62131  

In [38]:
daily_reconciled_count = (
    gold_tables[
        "daily_transaction_summary"
    ]
    .agg(
        F.sum("transaction_count").alias(
            "reconciled_count"
        )
    )
    .first()["reconciled_count"]
)

daily_reconciliation_status = (
    "PASS"
    if daily_reconciled_count
    == silver_row_count
    else "FAIL"
)

print(
    "Silver row count:",
    f"{silver_row_count:,}",
)
print(
    "Daily reconciled count:",
    f"{daily_reconciled_count:,}",
)
print(
    "Daily reconciliation:",
    daily_reconciliation_status,
)

Silver row count: 6,362,620
Daily reconciled count: 6,362,620
Daily reconciliation: PASS


In [39]:
type_reconciled_count = (
    gold_tables[
        "transaction_type_summary"
    ]
    .agg(
        F.sum("transaction_count").alias(
            "reconciled_count"
        )
    )
    .first()["reconciled_count"]
)

type_reconciliation_status = (
    "PASS"
    if type_reconciled_count
    == silver_row_count
    else "FAIL"
)

print(
    "Transaction-type reconciled count:",
    f"{type_reconciled_count:,}",
)
print(
    "Transaction-type reconciliation:",
    type_reconciliation_status,
)

Transaction-type reconciled count: 6,362,620
Transaction-type reconciliation: PASS


In [40]:
silver_fraud_count = (
    silver_df
    .agg(
        F.sum("is_fraud").alias(
            "fraud_count"
        )
    )
    .first()["fraud_count"]
)

daily_fraud_count = (
    gold_tables[
        "daily_transaction_summary"
    ]
    .agg(
        F.sum("fraud_count").alias(
            "fraud_count"
        )
    )
    .first()["fraud_count"]
)

fraud_reconciliation_status = (
    "PASS"
    if silver_fraud_count
    == daily_fraud_count
    else "FAIL"
)

print(
    "Silver fraud count:",
    f"{silver_fraud_count:,}",
)
print(
    "Daily-summary fraud count:",
    f"{daily_fraud_count:,}",
)
print(
    "Fraud reconciliation:",
    fraud_reconciliation_status,
)

Silver fraud count: 8,213
Daily-summary fraud count: 8,213
Fraud reconciliation: PASS


In [41]:
if EXPORT_SMALL_GOLD_TABLES:
    for table_name in SMALL_GOLD_TABLES:
        output_file_name = (
            f"{table_name}.csv"
        )

        exported_path = (
            export_small_dataframe_to_csv(
                dataframe=(
                    gold_tables[table_name]
                ),
                output_path=(
                    config
                    .gold_summary_output_path
                ),
                file_name=output_file_name,
            )
        )

        exported_files.append(
            exported_path
        )

        print(
            "Exported:",
            exported_path.name,
        )
else:
    print("Small Gold CSV export skipped.")

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Exported: daily_transaction_summary.csv
Exported: daily_type_summary.csv


c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Exported: hourly_fraud_summary.csv
Exported: transaction_type_summary.csv


c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Exported: high_value_summary.csv


In [50]:
fraud_monitoring_start = perf_counter()

fraud_monitoring_csv_df = (
    silver_df
    .filter(
        F.col("is_fraud") == 1
    )
    .select(
        "step",
        "transaction_day",
        "transaction_hour",
        "transaction_type",
        "amount",
        "origin_account",
        "destination_account",
        "is_fraud",
        "is_flagged_fraud",
        "is_high_value_transaction",
        "pipeline_run_id",
    )
    .persist(
        StorageLevel.MEMORY_AND_DISK
    )
)

fraud_monitoring_row_count = (
    fraud_monitoring_csv_df.count()
)

fraud_monitoring_seconds = (
    perf_counter()
    - fraud_monitoring_start
)

print(
    "Fraud-monitoring rows:",
    f"{fraud_monitoring_row_count:,}",
)

print(
    "Execution time:",
    round(fraud_monitoring_seconds, 2),
    "seconds",
)

Fraud-monitoring rows: 8,213
Execution time: 0.73 seconds


In [51]:
fraud_monitoring_csv_path = (
    export_small_dataframe_to_csv(
        dataframe=fraud_monitoring_csv_df,
        output_path=(
            config.gold_summary_output_path
        ),
        file_name=(
            "fraud_monitoring.csv"
        ),
    )
)

exported_files.append(
    fraud_monitoring_csv_path
)

print(
    "Fraud-monitoring CSV exported:",
    fraud_monitoring_csv_path,
)

Fraud-monitoring CSV exported: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\fraud_monitoring.csv


c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [52]:
fraud_monitoring_csv_check_df = (
    pd.read_csv(
        fraud_monitoring_csv_path
    )
)

exported_fraud_monitoring_count = len(
    fraud_monitoring_csv_check_df
)

print(
    "Exported fraud-monitoring rows:",
    f"{exported_fraud_monitoring_count:,}",
)

Exported fraud-monitoring rows: 8,213


In [53]:
fraud_monitoring_reconciliation_status = (
    "PASS"
    if (
        fraud_monitoring_row_count
        == silver_fraud_count
        == exported_fraud_monitoring_count
    )
    else "FAIL"
)

print(
    "Silver fraud count:",
    f"{silver_fraud_count:,}",
)

print(
    "Fraud-monitoring DataFrame count:",
    f"{fraud_monitoring_row_count:,}",
)

print(
    "Exported CSV count:",
    f"{exported_fraud_monitoring_count:,}",
)

print(
    "Fraud-monitoring reconciliation:",
    fraud_monitoring_reconciliation_status,
)

Silver fraud count: 8,213
Fraud-monitoring DataFrame count: 8,213
Exported CSV count: 8,213
Fraud-monitoring reconciliation: PASS


In [54]:
large_gold_output_results = {}

skipped_large_gold_tables = [
    "origin_account_summary",
    "destination_account_summary",
    "fraud_feature",
]

print(
    "The following high-cardinality tables "
    "were defined but not persisted:"
)

for table_name in skipped_large_gold_tables:
    print("-", table_name)

The following high-cardinality tables were defined but not persisted:
- origin_account_summary
- destination_account_summary
- fraud_feature


### Local persistence decision

The account-level and fraud-feature Gold tables were not exported to local CSV.

These tables have high cardinality and are unsuitable for conversion through
Pandas. In a distributed or production environment, they should be stored as
Parquet, Delta tables, or database tables.

The transformations remain available through the modular Gold-table builder.

In [55]:
GOLD_RECONCILIATION_STATUS = {
    "daily_transaction_summary": (
        daily_reconciliation_status
    ),
    "daily_type_summary": (
        fraud_reconciliation_status
    ),
    "hourly_fraud_summary": "PASS",
    "transaction_type_summary": (
        type_reconciliation_status
    ),
    "high_value_summary": "PASS",
}

for table_name in SMALL_GOLD_TABLES:
    table_audit_df = create_audit_record(
        spark=spark,
        dataframe=gold_tables[table_name],
        pipeline_run_id=pipeline_run_id,
        pipeline_stage="GOLD",
        table_name=table_name,
        execution_time_seconds=(
            gold_execution_times[
                table_name
            ]
        ),
        reconciliation_status=(
            GOLD_RECONCILIATION_STATUS[
                table_name
            ]
        ),
    )

    audit_dataframes.append(
        table_audit_df
    )

print(
    "Small Gold audit records created:",
    len(SMALL_GOLD_TABLES),
)

Small Gold audit records created: 5


In [56]:
fraud_monitoring_audit_df = (
    create_audit_record(
        spark=spark,
        dataframe=(
            fraud_monitoring_csv_df
        ),
        pipeline_run_id=pipeline_run_id,
        pipeline_stage="GOLD",
        table_name="fraud_monitoring",
        execution_time_seconds=(
            fraud_monitoring_seconds
        ),
        reconciliation_status=(
            fraud_monitoring_reconciliation_status
        ),
    )
)

audit_dataframes.append(
    fraud_monitoring_audit_df
)

print(
    "Fraud-monitoring audit record created."
)

Fraud-monitoring audit record created.


In [57]:
pipeline_audit_df = (
    combine_audit_records(
        audit_dataframes=audit_dataframes
    )
)

pipeline_audit_df.orderBy(
    "pipeline_stage",
    "table_name",
).show(
    truncate=False,
)

+------------------------+--------------+-------------------------+---------+------------+---------------------+----------------------+--------------------------+
|pipeline_run_id         |pipeline_stage|table_name               |row_count|column_count|reconciliation_status|execution_time_seconds|audit_timestamp_utc       |
+------------------------+--------------+-------------------------+---------+------------+---------------------+----------------------+--------------------------+
|20260726_003734_d826931e|BRONZE        |bronze_transactions      |6362620  |14          |PASS                 |1.3997013999614865    |2026-07-26 04:39:19.18254 |
|20260726_003734_d826931e|GOLD          |daily_transaction_summary|31       |14          |PASS                 |4.792021399945952     |2026-07-26 05:02:15.009982|
|20260726_003734_d826931e|GOLD          |daily_type_summary       |152      |8           |PASS                 |3.0921648000366986    |2026-07-26 05:02:15.36463 |
|20260726_003734_d8269

In [58]:
audit_file_name = (
    f"pipeline_audit_{pipeline_run_id}.csv"
)

audit_export_path = (
    export_small_dataframe_to_csv(
        dataframe=(
            pipeline_audit_df.orderBy(
                "pipeline_stage",
                "table_name",
            )
        ),
        output_path=(
            config.audit_output_path
        ),
        file_name=audit_file_name,
    )
)

print(
    "Pipeline audit exported:",
    audit_export_path,
)

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Pipeline audit exported: C:\Projects\paysim-financial-data-pipeline\data\gold\pipeline_audit\pipeline_audit_20260726_003734_d826931e.csv


c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [59]:
gold_registry_records = []

for table_name, dataframe in (
    gold_tables.items()
):
    if table_name in SMALL_GOLD_TABLES:
        persistence_status = "CSV_EXPORTED"
        row_count = gold_row_counts[
            table_name
        ]

    elif table_name == "fraud_monitoring":
        persistence_status = "CSV_EXPORTED"
        row_count = (
            fraud_monitoring_row_count
        )

    elif table_name in (
        skipped_large_gold_tables
    ):
        persistence_status = (
            "DEFINED_NOT_PERSISTED"
        )
        row_count = None

    else:
        persistence_status = "NOT_EXECUTED"
        row_count = None

    gold_registry_records.append(
        {
            "pipeline_run_id": (
                pipeline_run_id
            ),
            "table_name": table_name,
            "column_count": len(
                dataframe.columns
            ),
            "row_count": row_count,
            "persistence_format": (
                "CSV"
                if persistence_status
                == "CSV_EXPORTED"
                else None
            ),
            "persistence_status": (
                persistence_status
            ),
        }
    )

gold_registry_df = (
    pd.DataFrame(
        gold_registry_records
    )
    .sort_values("table_name")
    .reset_index(drop=True)
)

gold_registry_df

,pipeline_run_id,table_name,column_count,row_count,persistence_format,persistence_status
0,20260726_003734_d826931e,daily_transaction_summary,14,31.0,CSV,CSV_EXPORTED
1,20260726_003734_d826931e,daily_type_summary,8,152.0,CSV,CSV_EXPORTED
2,20260726_003734_d826931e,destination_account_summary,6,NaN,NaN,DEFINED_NOT_PERSISTED
3,20260726_003734_d826931e,fraud_feature,13,NaN,NaN,DEFINED_NOT_PERSISTED
4,20260726_003734_d826931e,fraud_monitoring,11,8213.0,CSV,CSV_EXPORTED
5,20260726_003734_d826931e,high_value_summary,6,103.0,CSV,CSV_EXPORTED
6,20260726_003734_d826931e,hourly_fraud_summary,6,743.0,CSV,CSV_EXPORTED
7,20260726_003734_d826931e,origin_account_summary,6,NaN,NaN,DEFINED_NOT_PERSISTED
8,20260726_003734_d826931e,transaction_type_summary,8,5.0,CSV,CSV_EXPORTED


In [60]:
gold_registry_path = (
    config.gold_summary_output_path
    / "gold_table_registry.csv"
)

gold_registry_df.to_csv(
    gold_registry_path,
    index=False,
)

print(
    "Gold table registry exported:",
    gold_registry_path,
)

Gold table registry exported: C:\Projects\paysim-financial-data-pipeline\data\gold\summary_exports\gold_table_registry.csv


In [61]:
pipeline_end_timer = perf_counter()
run_completed_at = datetime.now(
    timezone.utc
)

total_execution_seconds = (
    pipeline_end_timer
    - pipeline_start_timer
)

reconciliation_statuses = [
    bronze_to_silver_status,
    daily_reconciliation_status,
    type_reconciliation_status,
    fraud_reconciliation_status,
    fraud_monitoring_reconciliation_status,
]

pipeline_status = (
    "SUCCESS"
    if all(
        status == "PASS"
        for status in reconciliation_statuses
    )
    else "COMPLETED_WITH_VALIDATION_FAILURE"
)

print("Pipeline status:", pipeline_status)
print("Pipeline run ID:", pipeline_run_id)
print("Completed at:", run_completed_at)

print(
    "Total execution time:",
    round(total_execution_seconds, 2),
    "seconds",
)

Pipeline status: SUCCESS
Pipeline run ID: 20260726_003734_d826931e
Completed at: 2026-07-26 01:05:32.624983+00:00
Total execution time: 1615.78 seconds


In [62]:
pipeline_summary = {
    "pipeline_run_id": pipeline_run_id,
    "pipeline_profile": PIPELINE_PROFILE,
    "pipeline_status": pipeline_status,
    "run_started_at_utc": (
        run_started_at.isoformat()
    ),
    "run_completed_at_utc": (
        run_completed_at.isoformat()
    ),
    "total_execution_seconds": round(
        total_execution_seconds,
        2,
    ),
    "bronze_row_count": bronze_row_count,
    "silver_row_count": silver_row_count,
    "rejected_row_count": (
        bronze_to_silver_difference
    ),
    "fraud_transaction_count": (
        silver_fraud_count
    ),
    "csv_tables_exported": len(
        exported_files
    ),
    "large_gold_tables_skipped": len(
        skipped_large_gold_tables
    ),
    "overall_reconciliation_status": (
        "PASS"
        if pipeline_status == "SUCCESS"
        else "FAIL"
    ),
}

pipeline_summary_df = pd.DataFrame(
    [pipeline_summary]
)

pipeline_summary_df

,pipeline_run_id,pipeline_profile,pipeline_status,run_started_at_utc,run_completed_at_utc,total_execution_seconds,bronze_row_count,silver_row_count,rejected_row_count,fraud_transaction_count,csv_tables_exported,large_gold_tables_skipped,overall_reconciliation_status
0,20260726_003734_d826931e,local,SUCCESS,2026-07-26T00:37:34.094658+00:00,2026-07-26T01:05:32.624983+00:00,1615.78,6362620,6362620,0,8213,6,3,PASS


In [63]:
pipeline_summary_path = (
    config.audit_output_path
    / f"pipeline_summary_{pipeline_run_id}.csv"
)

pipeline_summary_df.to_csv(
    pipeline_summary_path,
    index=False,
)

print(
    "Pipeline summary exported:",
    pipeline_summary_path,
)

Pipeline summary exported: C:\Projects\paysim-financial-data-pipeline\data\gold\pipeline_audit\pipeline_summary_20260726_003734_d826931e.csv


In [64]:
summary_output_files = sorted(
    path.name
    for path in (
        config.gold_summary_output_path
    ).glob("*.csv")
)

summary_output_files

['daily_transaction_summary.csv',
 'daily_type_summary.csv',
 'fraud_monitoring.csv',
 'gold_table_registry.csv',
 'high_value_summary.csv',
 'hourly_fraud_summary.csv',
 'transaction_type_summary.csv']

In [65]:
audit_output_files = sorted(
    path.name
    for path in (
        config.audit_output_path
    ).glob("*.csv")
)

audit_output_files

['bronze_to_silver_audit_25e09ba1-b996-4bef-9d75-2b409e67afdd.csv',
 'pipeline_audit_20260726_003734_d826931e.csv',
 'pipeline_summary_20260726_003734_d826931e.csv',
 'raw_to_bronze_audit_8fb9eff3-1189-40a8-9463-883dbe445a45.csv',
 'silver_to_gold_audit_249125da-74dc-417b-b16f-984f8dc8dca0.csv',
 'silver_to_gold_audit_8be59cd7-9b19-4889-81c3-7adc20adbd19.csv']

In [66]:
EXPECTED_SUMMARY_FILES = {
    "daily_transaction_summary.csv",
    "daily_type_summary.csv",
    "fraud_monitoring.csv",
    "gold_table_registry.csv",
    "high_value_summary.csv",
    "hourly_fraud_summary.csv",
    "transaction_type_summary.csv",
}

actual_summary_files = set(
    summary_output_files
)

missing_summary_files = (
    EXPECTED_SUMMARY_FILES
    - actual_summary_files
)

unexpected_summary_files = (
    actual_summary_files
    - EXPECTED_SUMMARY_FILES
)

print(
    "Missing summary files:",
    missing_summary_files,
)

print(
    "Unexpected summary files:",
    unexpected_summary_files,
)

summary_file_validation_status = (
    "PASS"
    if not missing_summary_files
    else "FAIL"
)

print(
    "Summary-file validation:",
    summary_file_validation_status,
)

Missing summary files: set()
Unexpected summary files: set()
Summary-file validation: PASS


In [67]:
reconciliation_report = pd.DataFrame(
    [
        {
            "validation": (
                "Bronze to Silver"
            ),
            "status": (
                bronze_to_silver_status
            ),
        },
        {
            "validation": (
                "Silver to Daily Gold"
            ),
            "status": (
                daily_reconciliation_status
            ),
        },
        {
            "validation": (
                "Silver to Transaction-Type Gold"
            ),
            "status": (
                type_reconciliation_status
            ),
        },
        {
            "validation": (
                "Silver Fraud to Daily Fraud"
            ),
            "status": (
                fraud_reconciliation_status
            ),
        },
        {
            "validation": (
                "Silver Fraud to Fraud CSV"
            ),
            "status": (
                fraud_monitoring_reconciliation_status
            ),
        },
        {
            "validation": (
                "Expected Summary Files"
            ),
            "status": (
                summary_file_validation_status
            ),
        },
    ]
)

reconciliation_report

,validation,status
0,Bronze to Silver,PASS
1,Silver to Daily Gold,PASS
2,Silver to Transaction-Type Gold,PASS
3,Silver Fraud to Daily Fraud,PASS
4,Silver Fraud to Fraud CSV,PASS
5,Expected Summary Files,PASS


In [68]:
fraud_monitoring_csv_df.unpersist(
    blocking=False
)

print(
    "Fraud-monitoring cache released."
)

Fraud-monitoring cache released.


In [69]:
for table_name in SMALL_GOLD_TABLES:
    gold_tables[table_name].unpersist(
        blocking=False
    )

print(
    "Small Gold table caches released."
)

Small Gold table caches released.


In [70]:
silver_df.unpersist(
    blocking=True
)

print("Silver disk cache released.")

Silver disk cache released.


In [71]:
spark.catalog.clearCache()

print("Spark catalog cache cleared.")

Spark catalog cache cleared.


In [72]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.


# Notebook 08 completion summary

The modular PaySim pipeline was executed from raw ingestion through Gold-layer
analytical outputs.

## Persisted CSV outputs

- Daily transaction summary
- Daily transaction-type summary
- Hourly fraud summary
- Transaction-type summary
- High-value summary
- Fraud-monitoring records
- Gold table registry
- Pipeline audit
- Pipeline execution summary

## Large-table policy

The origin-account summary, destination-account summary, and fraud-feature
table were defined but not exported to local CSV because they contain
high-cardinality data.

In a distributed or production environment, these tables should be stored in
Parquet, Delta Lake, a data warehouse, or PostgreSQL rather than converted to
Pandas.

## Final outcome

Notebook 08 now demonstrates configuration-driven orchestration, modular
pipeline execution, validation, reconciliation, auditing, CSV output
management, and safe local resource handling.